# Legendaryvowels 단계별 테스트

아래 설정 셀을 먼저 실행한 뒤 확인하려는 셀만 실행합니다.

- **문장 축 / STT 테스트**: Deepgram 전사와 단어 시간 확인
- **문장 축 / 전체 평가**: STT → 문장 정렬 → 문장 점수 및 피드백
- **글자 축 / LPC 테스트**: STT 없이 LPC, F1/F2/F3, envelope 확인
- **글자 축 / LPC 평가**: 정답 LPC reference와 비교

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import JSON, display

current = Path.cwd().resolve()
package_directory = (
    current
    if current.name == 'Legendaryvowels'
    else current / 'sushisite' / 'sushi-fast' / 'Legendaryvowels'
)
if not package_directory.exists():
    raise FileNotFoundError(f'Legendaryvowels 폴더를 찾을 수 없습니다: {package_directory}')

sys.path.insert(0, str(package_directory.parent))

from Legendaryvowels.services.pronunciation.feature import LPCFeatureExtractor
from Legendaryvowels.services.pronunciation.lpc_evaluator import LPCEvaluator
from Legendaryvowels.services.stt.factory import create_stt_service
from Legendaryvowels.services.sentence.service import analyze_voice
from Legendaryvowels.schemas import ProductMode

AUDIO_PATH = package_directory / 'sample.m4a'
TARGET_TEXT = '안녕하세요, 고 발표를 시작하겠습니다.'
VOWEL = '아'

print('audio:', AUDIO_PATH)
print('reference:', package_directory / 'reference_lpc' / f'{VOWEL}.json')

## 1. 문장 축: STT만 테스트

Deepgram API를 실제 호출합니다. `transcript`, 단어별 시작·종료 시간과 confidence를 확인합니다.

## 2. 문장 축: 전체 문장 평가

STT → 문장 정렬 → 규칙 점수와 피드백을 실행합니다. LPC는 계산하지 않습니다.

In [ ]:
sentence_result = analyze_voice(
    audio_path=str(AUDIO_PATH),
    mode=ProductMode.EDUCATION,
    session_id='notebook-session',
    attempt_id='notebook-attempt',
    target_text=TARGET_TEXT,
)
display(JSON(sentence_result.model_dump(by_alias=True, mode='json'), expanded=False))

In [ ]:
stt_result = create_stt_service().transcribe(str(AUDIO_PATH))
display(JSON(stt_result.model_dump(mode='json'), expanded=True))

## 3. 글자 축: LPC만 테스트

STT API를 호출하지 않습니다. 오디오에서 LPC envelope와 F1/F2/F3를 추출합니다.

In [ ]:
lpc_result = LPCFeatureExtractor().extract(str(AUDIO_PATH))
print('formants:', lpc_result.formants)
print('envelope points:', len(lpc_result.envelope.freq))
display(JSON(lpc_result.model_dump(mode='json'), expanded=False))

## 4. 글자 축: LPC 평가 테스트

`reference_lpc/{VOWEL}.json`이 먼저 생성되어 있어야 합니다. STT는 호출하지 않습니다.

In [ ]:
if 'lpc_result' not in globals():
    lpc_result = LPCFeatureExtractor().extract(str(AUDIO_PATH))

evaluation = LPCEvaluator().evaluate(VOWEL, lpc_result)
display(JSON(evaluation.model_dump(mode='json'), expanded=True))

## 5. 글자 축: 최종 word JSON 넣고 LPC 그래프 시각화

VS Code/WSL에서는 파일 선택 위젯이 안 뜨는 경우가 있습니다. 그래서 기본은 `response_*.json` 또는 `word_result*.json` 중 가장 최신 파일을 자동으로 잡고, 필요하면 `WORD_RESULT_JSON_PATH`만 직접 바꿔 실행합니다.


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

# 1) 직접 지정하고 싶으면 여기를 바꾸세요.
# 예: WORD_RESULT_JSON_PATH = package_directory / 'response_1784426098375.json'
WORD_RESULT_JSON_PATH = None


def _find_latest_word_json():
    candidates = []
    for pattern in ('response_*.json', 'word_result*.json'):
        candidates.extend(package_directory.glob(pattern))
    candidates = [path for path in candidates if path.is_file()]
    if not candidates:
        raise FileNotFoundError(
            f'{package_directory} 안에서 response_*.json 또는 word_result*.json 파일을 찾지 못했습니다.\n'
            'WORD_RESULT_JSON_PATH = package_directory / "파일명.json" 형태로 직접 지정해주세요.'
        )
    return max(candidates, key=lambda path: path.stat().st_mtime)


def _load_word_result_json(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'word 결과 JSON 파일을 찾을 수 없습니다: {path}')
    payload = json.loads(path.read_text(encoding='utf-8'))

    if 'graph' in payload:
        return payload
    if 'evaluation' in payload and 'graph' in payload['evaluation']:
        return payload['evaluation']

    raise ValueError('최상위 graph 또는 evaluation.graph가 있는 word 결과 JSON이 필요합니다.')


def _point_value(point, snake_name, camel_name):
    if snake_name in point:
        return point[snake_name]
    return point[camel_name]


def _series(points):
    return (
        [_point_value(point, 'frequency_hz', 'frequencyHz') for point in points],
        [_point_value(point, 'magnitude_db', 'magnitudeDb') for point in points],
    )


def _formants(payload, snake_name, camel_name):
    return payload.get(snake_name) or payload.get(camel_name) or {}


def _mark_formants(formants, label_prefix, color):
    for name, frequency in sorted(formants.items()):
        if frequency is None:
            continue
        plt.axvline(frequency, color=color, alpha=0.22, linewidth=1)
        plt.text(
            frequency,
            1.5,
            f'{label_prefix} {name}\n{frequency:.0f} Hz',
            color=color,
            fontsize=9,
            ha='center',
            va='bottom',
        )


if WORD_RESULT_JSON_PATH is None:
    WORD_RESULT_JSON_PATH = _find_latest_word_json()

word_result_json = _load_word_result_json(WORD_RESULT_JSON_PATH)
user_points = word_result_json['graph']['user']
target_points = word_result_json['graph']['target']
user_freq, user_amp = _series(user_points)
target_freq, target_amp = _series(target_points)
user_formants = _formants(word_result_json, 'user_formants', 'userFormants')
target_formants = _formants(word_result_json, 'target_formants', 'targetFormants')

plt.figure(figsize=(12, 5))
plt.plot(target_freq, target_amp, label='target reference LPC', linewidth=2.4)
plt.plot(user_freq, user_amp, label='user LPC', linewidth=2.0, alpha=0.88)
_mark_formants(target_formants, 'target', 'tab:blue')
_mark_formants(user_formants, 'user', 'tab:orange')

plt.title(f'LPC envelope comparison from JSON - {WORD_RESULT_JSON_PATH.name}')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Magnitude (dB, normalized)')
plt.xlim(0, 5000)
plt.grid(True, alpha=0.25)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print('json:', WORD_RESULT_JSON_PATH)
print('score:', word_result_json.get('score'))
print('distance:', word_result_json.get('distance'))
print('feedback:', word_result_json.get('feedback'))
print('delta:', word_result_json.get('delta'))
print('user points:', len(user_points))
print('target points:', len(target_points))